# ⚡ Fine-Tuning with LoRA — Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shubh-vedi/awesome-genai-toolkit/blob/main/notebooks/fine-tuning/lora-quickstart.ipynb)

Fine-tune an LLM on your own data using LoRA (Low-Rank Adaptation) with Unsloth for 2x speed.

**What you'll learn:**
- What LoRA and QLoRA are
- Setting up Unsloth for efficient fine-tuning
- Preparing training data
- Running fine-tuning
- Testing and saving the model

**Requirements:** Colab with T4 GPU (free tier works!)

## 📦 Installation

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

## 1️⃣ Load a Pre-trained Model

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load Llama 3.2 1B (small enough for free Colab)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=2048,
    dtype=None,        # Auto-detect
    load_in_4bit=True, # QLoRA — saves memory
)

print(f"Model loaded! Parameters: {model.num_parameters():,}")

## 2️⃣ Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                         # LoRA rank (higher = more capacity, more memory)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,               # 0 is optimized for speed
    bias="none",
    use_gradient_checkpointing="unsloth",  # Saves 30% memory
)

# Check trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 3️⃣ Prepare Training Data

In [ ]:
from datasets import Dataset

# Example: Fine-tune for customer support
training_data = [
    {"instruction": "How do I reset my password?",
     "response": "To reset your password: 1) Go to the login page, 2) Click 'Forgot Password', 3) Enter your email, 4) Check your inbox for a reset link, 5) Create a new password with at least 8 characters."},
    {"instruction": "What are your business hours?",
     "response": "Our business hours are Monday through Friday, 9 AM to 6 PM EST. We also offer limited weekend support from 10 AM to 2 PM EST."},
    {"instruction": "How do I cancel my subscription?",
     "response": "To cancel your subscription: 1) Log into your account, 2) Go to Settings > Billing, 3) Click 'Cancel Subscription', 4) Confirm cancellation. Your access continues until the end of the billing period."},
    {"instruction": "Do you offer refunds?",
     "response": "Yes, we offer a 30-day money-back guarantee. To request a refund, contact support@example.com with your order number. Refunds are processed within 5-7 business days."},
    {"instruction": "How do I contact support?",
     "response": "You can reach our support team through: 1) Email: support@example.com, 2) Live chat on our website (24/7), 3) Phone: 1-800-SUPPORT (Mon-Fri 9-6 EST). Average response time is under 2 hours."},
]

# Format for chat template
def format_for_training(item):
    messages = [
        {"role": "system", "content": "You are a helpful customer support agent. Be concise and friendly."},
        {"role": "user", "content": item["instruction"]},
        {"role": "assistant", "content": item["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_for_training)
print(f"Training examples: {len(dataset)}")
print(f"\nSample:\n{dataset[0]['text'][:300]}...")

## 4️⃣ Train!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=30,           # Increase for real training (100+)
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        output_dir="outputs",
        seed=42,
    ),
)

# Train
stats = trainer.train()
print(f"\n✅ Training complete! Loss: {stats.training_loss:.4f}")

## 5️⃣ Test the Fine-Tuned Model

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

# Test with a new question
messages = [
    {"role": "system", "content": "You are a helpful customer support agent. Be concise and friendly."},
    {"role": "user", "content": "How do I upgrade my plan?"},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

output = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

## 6️⃣ Save & Export

In [ ]:
# Save LoRA adapters (small, fast)
model.save_pretrained("my-support-bot-lora")
tokenizer.save_pretrained("my-support-bot-lora")
print("✅ LoRA adapters saved!")

# Optional: Save to Hugging Face Hub
# model.push_to_hub("your-username/my-support-bot")

# Optional: Export to GGUF for Ollama/llama.cpp
# model.save_pretrained_gguf("my-support-bot-gguf", quantization_method="q4_k_m")

## 🎯 Key Concepts

| Concept | Description |
|---------|-------------|
| **LoRA** | Trains small adapter layers instead of full model (1-3% of params) |
| **QLoRA** | LoRA + 4-bit quantization = fine-tune on consumer GPUs |
| **Unsloth** | 2x faster training, 60% less memory |
| **SFT** | Supervised Fine-Tuning on instruction/response pairs |
| **GGUF** | Export format for local inference (Ollama, LM Studio) |

## Next Steps
- 📄 [LoRA Paper](../resources/papers.md) — Understand the theory
- 📓 [LangChain Basics](../langchain-basics.ipynb) — Use your fine-tuned model in apps
- 🛠️ [Tools Reference](../resources/tools.md) — Deployment options